# 00 — Data Preparation

Loads raw OHLCV data for all B3 tickers, computes technical features, runs distribution tests,
removes bad features, applies Triple Barrier labeling, and saves processed parquets to disk.

In [ ]:
import sys, warnings
from pathlib import Path
warnings.filterwarnings("ignore")
sys.path.insert(0, str(Path("../.." ).resolve()))
sys.path.insert(0, str(Path(".").resolve()))

import numpy as np
import pandas as pd
from tqdm.notebook import tqdm

from config.experiment_config import TICKERS, FEATURE_CONFIG, LABELING_CONFIG
from src.data_loader import load_raw, save_processed, available_tickers
from src.feature_engineering import compute_features, select_features
from src.labeling import label_asset, label_distribution

DATA_DIR = Path("../../data")
print(f"Available tickers: {available_tickers(DATA_DIR)}")

## 1. Load Raw Data

In [ ]:
raw_data = {}
for ticker in tqdm(TICKERS, desc="Loading"):
    try:
        raw_data[ticker] = load_raw(ticker, DATA_DIR)
        df = raw_data[ticker]
        print(f"  {ticker}: {len(df)} rows  {df.index[0].date()} \u2192 {df.index[-1].date()}")
    except Exception as e:
        print(f"  {ticker}: ERROR \u2014 {e}")

print(f"\nLoaded {len(raw_data)}/{len(TICKERS)} tickers")

## 2. Compute Features

In [ ]:
all_features = {}
for ticker, df in tqdm(raw_data.items(), desc="Computing features"):
    try:
        feat = compute_features(df, FEATURE_CONFIG)
        all_features[ticker] = feat
    except Exception as e:
        print(f"  {ticker}: FEATURE ERROR \u2014 {e}")

# Show feature list
sample = next(iter(all_features.values()))
print(f"\nFeatures computed ({len(sample.columns)}):")
for col in sample.columns:
    print(f"  {col}")

## 3. Distribution Tests & Feature Selection

In [ ]:
selected_cols, report = select_features(all_features, FEATURE_CONFIG)

print("=== Feature Selection Report ===")
print(report.to_string())
print(f"\nSelected: {len(selected_cols)} / {len(report)} features")
print(f"Removed:  {len(report) - len(selected_cols)}")
print("\nRemoved features:")
removed = report[~report["passed_distribution"]]
for feat, row in removed.iterrows():
    print(f"  {feat}: {row['removed_reason']}")

In [ ]:
for ticker in all_features:
    all_features[ticker] = all_features[ticker][selected_cols]

print(f"Feature matrix shape per asset: {next(iter(all_features.values())).shape}")
print(f"Selected features: {selected_cols}")

## 4. Triple Barrier Labeling

In [ ]:
all_labels = {}
label_stats = {}

for ticker, df in tqdm(raw_data.items(), desc="Labeling"):
    try:
        labels = label_asset(df, LABELING_CONFIG)
        all_labels[ticker] = labels
        dist = label_distribution(labels)
        label_stats[ticker] = dist
        print(f"  {ticker}: {len(labels)} labels | {dist.to_dict()}")
    except Exception as e:
        print(f"  {ticker}: LABEL ERROR \u2014 {e}")

In [ ]:
stats_df = pd.DataFrame(label_stats).T
stats_df.columns.name = "class"
print("\nLabel distribution (%) per ticker:")
print(stats_df.to_string())
print(f"\nMean: {stats_df.mean().round(1).to_dict()}")

## 5. Align Features and Labels & Save

In [ ]:
saved = 0
skipped = 0
for ticker in tqdm(TICKERS, desc="Saving"):
    if ticker not in all_features or ticker not in all_labels:
        skipped += 1
        continue
    feat   = all_features[ticker]
    labels = all_labels[ticker]
    # Align to common index
    common = feat.index.intersection(labels.index)
    feat   = feat.loc[common].dropna()
    labels = labels.loc[feat.index]

    save_processed(ticker, feat, labels)
    saved += 1

print(f"\nSaved: {saved}  |  Skipped: {skipped}")
print("Processed data saved to data/processed/ and data/labels/")

## 6. Quick Sanity Check

In [ ]:
from src.data_loader import load_processed, load_labels

ticker_check = TICKERS[0]
feat_check   = load_processed(ticker_check)
lab_check    = load_labels(ticker_check)
print(f"Ticker: {ticker_check}")
print(f"Features: {feat_check.shape}  |  Labels: {lab_check.shape}")
print(f"Features NaN: {feat_check.isna().sum().sum()}")
print(f"Label counts: {lab_check.value_counts().sort_index().to_dict()}")
feat_check.tail(3)